In [5]:
# Step 1: Setup
!pip install nltk

In [63]:
#translate language to english
!pip install deep-translator

In [25]:
!pip install emoji

   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   -------------------------------- ----- 524.3/608.4 kB 381.0 kB/s eta 0:00:01
   -------------------------------------- 608.4/608.4 kB 388.7 kB/s eta 0:00:00


In [3]:
!pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ------------------------------------- 981.5/981.5 kB 23.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993313 sha256=6bc9bb4b9626fef23285b48a3a34d0313ea44d7026c965f84f8860936aab8972
  Stored in directory: c:\users\asus\appdata\local\pip\cache\wheels\c1\67\88\e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [5]:
import pandas as pd
import re
import string
import emoji
from langdetect import detect
from deep_translator import GoogleTranslator
from tqdm import tqdm

tqdm.pandas()  # shows translation progress

In [11]:
df_reviews = pd.read_csv(r"C:\laragon\www\web\lokalkita\data\review_raw.csv")

# Display first few rows
df_reviews.head()


,Title,Username,Review
0,A’moss,AIN NADIAH,We have our Anniversary Trip here for 2D 1N.Th...
1,A’moss,Nur Suhaira,"An Unforgettable Escape - Homely, Breathtaking..."
2,A’moss,Atiq Safian,If you’re looking for a peaceful escape to rel...
3,A’moss,Scott Tian,Have a lovely stayed on 3/1 on my birthday .. ...
4,A’moss,Alfie Solomons,I’ve finally made my way to this elusive glamp...


In [13]:
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Remove emoji
    text = emoji.replace_emoji(text, replace='')

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove numbers and punctuations
    text = text.translate(str.maketrans('', '', string.punctuation + string.digits))

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [15]:
df_reviews['clean_review'] = df_reviews['Review'].apply(clean_text)

In [17]:
import os
from tqdm import tqdm
from deep_translator import GoogleTranslator
from langdetect import detect

checkpoint_path = "checkpoint_translated.csv"

# 🔁 Load from checkpoint if exists
if os.path.exists(checkpoint_path):
    print("🔁 Loading from checkpoint...")
    df_reviews = pd.read_csv(checkpoint_path)
else:
    print("🚀 Starting fresh translation...")

# 🧱 Make sure column exists
if 'translated_review' not in df_reviews.columns:
    df_reviews['translated_review'] = ""

# 🌍 Translate only missing ones
for i in tqdm(range(len(df_reviews)), desc="Translating Reviews"):
    text = df_reviews.loc[i, 'clean_review']

    # Skip if already translated
    if not pd.isna(df_reviews.loc[i, 'translated_review']) and str(df_reviews.loc[i, 'translated_review']).strip() != "":
        continue

    try:
        if isinstance(text, str) and text.strip() != "":
            lang = detect(text)
            
            # Skip translation if English or already detected as 'en'
            if lang.lower() == "en":
                df_reviews.loc[i, 'translated_review'] = text
                continue

            # Translate to English
            translated = GoogleTranslator(source='auto', target='en').translate(text)
            df_reviews.loc[i, 'translated_review'] = translated

    except Exception as e:
        print(f"⚠️ Skipping row {i} due to error: {e}")
        continue

    # 💾 Save progress every 100 rows
    if i % 100 == 0 and i > 0:
        df_reviews.to_csv(checkpoint_path, index=False)
        print(f"💾 Saved checkpoint at row {i}")

# Final save after loop
df_reviews.to_csv(checkpoint_path, index=False)
print("✅ Translation complete and saved to checkpoint_translated.csv")


🚀 Starting fresh translation...


Translating Reviews:   2%|▏         | 301/15590 [00:59<1:33:58,  2.71it/s]

💾 Saved checkpoint at row 300


Translating Reviews:   5%|▌         | 801/15590 [02:49<2:59:59,  1.37it/s]

💾 Saved checkpoint at row 800


...

💾 Saved checkpoint at row 14300


Translating Reviews:  92%|█████████▏| 14401/15590 [1:43:29<06:29,  3.05it/s]

💾 Saved checkpoint at row 14400


Translating Reviews: 100%|██████████| 15590/15590 [1:45:20<00:00,  2.47it/s] 


✅ Translation complete and saved to checkpoint_translated.csv


In [19]:
def post_clean(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_reviews['final_review'] = df_reviews['translated_review'].apply(post_clean)


In [25]:
df_reviews = df_reviews[df_reviews['final_review'].str.strip() != ""]
df_reviews.reset_index(drop=True, inplace=True)

In [27]:
df_reviews.to_csv("cleaned_translated_reviews.csv", index=False)
print("✅ Translation + Cleaning Complete! Saved as cleaned_translated_reviews.csv")

✅ Translation + Cleaning Complete! Saved as cleaned_translated_reviews.csv
